# AgentDecompile Usage Validation

 > Local-first, credential-free validation notebook for the executable parts of `USAGE.md`.

## Safety Contract

- Everything is read-only by default.
- Shared-server workflows are documentation-only unless explicitly enabled.
- Version-control-sensitive tools such as `checkout-program` and `checkin-program` are probed only to confirm they fail clearly in unsupported local contexts.

## Coverage

- Kernel and package validation
- Usage-doc command inventory
- Local fixture generation using repository helpers
- Local MCP server startup and discovery
- Local binary import/open and inspection flows
- Export and raw tool calls where supported
- Explicit version-control behavior checks

## Notebook Strategy

This notebook translates the docs into a runnable local harness wherever possible.

- `USAGE.md` is treated as the source of truth for command intent.
- Shared-server examples stay visible for onboarding, but they are not executed in the default run.
- Results are collected into a single summary table so mismatches between documented and observed behavior are obvious.

The notebook prefers native Jupyter features where they help:

- top-level `await` for MCP calls
- rich markdown and HTML summaries
- widgets for execution flags
- explicit cleanup cells so reruns stay predictable

## Live Contract Notes

This notebook now sits alongside a stricter terminal-validated E2E contract suite for the local server path.

- The default live local `tools/list` surface is currently 36 tools.
- Hidden compatibility tools such as `manage-comments` are still callable even though they are not advertised by default.
- `switch-project` still routes to `open-project` as a compatibility alias, but it is intentionally hidden from the default advertised surface.
- The strict local contract suite asserts the observed JSON payloads for `open-project`, `import-binary`, `list-project-files`, `get-current-program`, `list-functions`, `search-symbols`, `get-references`, `list-imports`, `list-exports`, `checkout-status`, `sync-project`, comment round-trips, and export artifacts.
- The current observed local `change-processor` behavior on `tests/fixtures/test_x86_64` is a failure response that leaves the active program unchanged; the suite records that exact contract so regressions are visible.

See `tests/test_e2e_local_terminal_contracts.py` and `examples/mcp_responses/local_live_contract_test_x86_64.json` for the captured reference used by those assertions.

In [ ]:
import asyncio
import html
import json
import os
import re
import shutil
import socket
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Any

import nest_asyncio
from IPython.display import HTML, JSON, Markdown, display

nest_asyncio.apply()

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "USAGE.md").exists():
            return candidate
    raise RuntimeError(f"Could not locate repository root from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_PATH = REPO_ROOT / "examples" / "usage_validation.ipynb"
USAGE_PATH = REPO_ROOT / "USAGE.md"
TMP_ROOT = REPO_ROOT / "tmp" / "usage_validation"
TMP_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = []

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
src_path = REPO_ROOT / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

def as_json(data):
    display(JSON(data, expanded=False))

def show_note(title: str, body: str, color: str = "#2855aa") -> None:
    display(
        HTML(
            f"<div style='border-left:4px solid {color}; padding:0.75rem 1rem; margin:0.5rem 0; background:#f7f9fc;'>"
            f"<strong>{html.escape(title)}</strong><br>{body}</div>"
        )
    )

usage_text = USAGE_PATH.read_text(encoding="utf-8")
env_report = {
    "repo_root": str(REPO_ROOT),
    "notebook": str(NOTEBOOK_PATH),
    "python_executable": sys.executable,
    "python_version": sys.version.split()[0],
    "usage_md_exists": USAGE_PATH.exists(),
    "shared_repo_quick_sequence_present": "### Shared repository quick sequence with uvx" in usage_text,
    "ghidra_install_dir": os.environ.get("GHIDRA_INSTALL_DIR"),
    "project_path_env": os.environ.get("AGENT_DECOMPILE_PROJECT_PATH"),
}

show_note(
    "Environment ready",
    "Kernel helpers, top-level await support, repo root detection, and import paths are initialized.",
    color="#1f6f43",
)
as_json(env_report)

In [3]:
import ipywidgets as widgets

RUN_SHARED_SERVER_DOCS = widgets.Checkbox(value=False, description="Show shared-server notes")
RUN_VERSION_CONTROL_PROBES = widgets.Checkbox(value=True, description="Probe local checkout/checkin behavior")
VERBOSE_TOOL_OUTPUT = widgets.Checkbox(value=False, description="Verbose captured output")

def current_flags() -> dict[str, bool]:
    return {
        "show_shared_server_notes": bool(RUN_SHARED_SERVER_DOCS.value),
        "probe_version_control_behavior": bool(RUN_VERSION_CONTROL_PROBES.value),
        "verbose_tool_output": bool(VERBOSE_TOOL_OUTPUT.value),
    }

controls = widgets.VBox(
    [
        widgets.HTML("<h4 style='margin:0'>Execution Flags</h4><p style='margin:0'>These affect notebook display and the explicit local behavior probes.</p>"),
        RUN_SHARED_SERVER_DOCS,
        RUN_VERSION_CONTROL_PROBES,
        VERBOSE_TOOL_OUTPUT,
    ]
 )

display(controls)
as_json(current_flags())

<IPython.core.display.JSON object>

In [ ]:
def extract_command_lines(path: Path) -> list[str]:
    lines = path.read_text(encoding="utf-8").splitlines()
    interesting = []
    for line in lines:
        stripped = line.strip()
        if any(token in stripped for token in ["agentdecompile-cli", "agentdecompile-server", "uvx --from", "call_tool", "Invoke-McpTool"]):
            interesting.append(stripped)
    return interesting

usage_commands = extract_command_lines(USAGE_PATH)
usage_text = USAGE_PATH.read_text(encoding="utf-8")

command_summary = {
    "USAGE.md command lines": len(usage_commands),
    "shared repository quick sequence present": "### Shared repository quick sequence with uvx" in usage_text,
}

sample_commands = usage_commands[:12]
rows = "".join(
    f"<tr><td style='padding:0.35rem 0.5rem; vertical-align:top'>{index}</td><td style='padding:0.35rem 0.5rem'><code>{html.escape(command)}</code></td></tr>"
    for index, command in enumerate(sample_commands, start=1)
 )

display(Markdown("## Doc Command Inventory"))
as_json(command_summary)
display(
    HTML(
        "<table style='border-collapse:collapse; width:100%'>"
        "<thead><tr><th style='text-align:left; padding:0.35rem 0.5rem'>#</th><th style='text-align:left; padding:0.35rem 0.5rem'>Sample command</th></tr></thead>"
        f"<tbody>{rows}</tbody></table>"
    )
)

In [5]:
from tests.helpers import create_minimal_binary

FIXTURE_DIR = REPO_ROOT / "tests" / "fixtures"
SESSION_DIR = Path(tempfile.mkdtemp(prefix="usage_validation_", dir=str(TMP_ROOT)))
PROJECT_DIR = SESSION_DIR / "project"
OUTPUT_DIR = SESSION_DIR / "exports"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_BINARY = SESSION_DIR / "minimal_test_elf.bin"
create_minimal_binary(LOCAL_BINARY)

fixture_candidates = [
    FIXTURE_DIR / "test_x86_64",
    FIXTURE_DIR / "test_arm64",
    FIXTURE_DIR / "test_fat_binary",
    LOCAL_BINARY,
 ]
available_binaries = [path for path in fixture_candidates if path.exists()]
PRIMARY_BINARY = available_binaries[0]

SESSION_STATE = {
    "session_dir": str(SESSION_DIR),
    "project_dir": str(PROJECT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "primary_binary": str(PRIMARY_BINARY),
    "available_binaries": [str(path) for path in available_binaries],
}

show_note(
    "Fixture workspace ready",
    f"Primary local binary: <code>{html.escape(str(PRIMARY_BINARY))}</code>",
    color="#7a4a00",
)
as_json(SESSION_STATE)

<IPython.core.display.JSON object>

In [24]:
from agentdecompile_cli.bridge import AgentDecompileMcpClient

SERVER_PROCESS = None
SERVER_LOG_HANDLE = None
SERVER_INFO = {}
CURRENT_PROGRAM_PATH = None
TOOL_NAMES = []

TOOL_ALIASES = {
    "open": ["open-project", "open_project", "import-binary", "import_binary"],
    "open-project": ["open-project", "open_project", "import-binary", "import_binary"],
    "import-binary": ["import-binary", "import_binary"],
    "list-project-files": ["list-project-files", "list_project_files"],
    "get-current-program": ["get-current-program", "get_current_program"],
    "list-functions": ["list-functions", "list_functions"],
    "get-functions": ["get-functions", "get_functions"],
    "search-symbols-by-name": ["search-symbols-by-name", "search_symbols_by_name"],
    "get-references": ["get-references", "get_references", "references"],
    "export": ["export"],
    "list-imports": ["list-imports", "list_imports"],
    "list-exports": ["list-exports", "list_exports"],
    "manage-strings": ["manage-strings", "manage_strings"],
    "search-constants": ["search-constants", "search_constants"],
    "get-call-graph": ["get-call-graph", "get_call_graph"],
    "analyze-data-flow": ["analyze-data-flow", "analyze_data_flow"],
    "checkout-status": ["checkout-status", "checkout_status"],
    "checkout-program": ["checkout-program", "checkout_program"],
    "checkin-program": ["checkin-program", "checkin_program"],
}

def find_free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])

def resolve_live_tool_name(name: str) -> str:
    candidates = TOOL_ALIASES.get(name, [name, name.replace("-", "_"), name.replace("_", "-")])
    for candidate in candidates:
        if candidate in TOOL_NAMES:
            return candidate
    return name

def read_server_log_tail(lines: int = 60) -> str:
    log_path = SERVER_INFO.get("log_path")
    if not log_path:
        return ""
    path = Path(log_path)
    if not path.exists():
        return ""
    return "\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])

def start_local_server(port: int | None = None) -> dict[str, str]:
    global SERVER_PROCESS, SERVER_LOG_HANDLE, SERVER_INFO
    if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
        return SERVER_INFO
    chosen_port = find_free_port() if port is None else port
    runtime_project_dir = PROJECT_DIR / f"runtime_{chosen_port}"
    runtime_project_dir.mkdir(parents=True, exist_ok=True)
    log_path = SESSION_DIR / f"server_{chosen_port}.log"
    env = os.environ.copy()
    for key in [
        "AGENT_DECOMPILE_BACKEND_URL",
        "AGENT_DECOMPILE_MCP_SERVER_URL",
        "AGENT_DECOMPILE_GHIDRA_SERVER_USERNAME",
        "AGENT_DECOMPILE_GHIDRA_SERVER_PASSWORD",
        "AGENT_DECOMPILE_GHIDRA_SERVER_HOST",
        "AGENT_DECOMPILE_GHIDRA_SERVER_PORT",
        "AGENT_DECOMPILE_GHIDRA_SERVER_REPOSITORY",
        "AGENTDECOMPILE_SERVER_HOST",
        "AGENTDECOMPILE_SERVER_PORT",
        "AGENTDECOMPILE_SERVER_USERNAME",
        "AGENTDECOMPILE_SERVER_PASSWORD",
        "AGENTDECOMPILE_SERVER_REPOSITORY",
    ]:
        env.pop(key, None)
    cmd = [
        sys.executable,
        "-m",
        "agentdecompile_cli.server",
        "-t",
        "streamable-http",
        "--host",
        "127.0.0.1",
        "--port",
        str(chosen_port),
        "--project-path",
        str(runtime_project_dir),
        "--project-name",
        "usage_validation",
    ]
    SERVER_LOG_HANDLE = open(log_path, "w", encoding="utf-8")
    SERVER_PROCESS = subprocess.Popen(
        cmd,
        cwd=str(REPO_ROOT),
        stdout=SERVER_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    SERVER_INFO = {
        "port": str(chosen_port),
        "url": f"http://127.0.0.1:{chosen_port}/mcp/message",
        "log_path": str(log_path),
        "project_path": str(runtime_project_dir),
        "command": " ".join(cmd),
    }
    return SERVER_INFO

async def wait_for_server(url: str, timeout: int = 120) -> bool:
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is not None:
            break
        try:
            async with AgentDecompileMcpClient(url=url) as client:
                await client.list_tools()
                return True
        except Exception as exc:
            last_error = exc
            await asyncio.sleep(1)
    if last_error is not None:
        print(f"Last connection error: {last_error}")
    return False

async def list_tools_live(url: str):
    async with AgentDecompileMcpClient(url=url) as client:
        return await client.list_tools()

async def list_resources_live(url: str):
    async with AgentDecompileMcpClient(url=url) as client:
        return await client.list_resources()

async def call_tool_live(url: str, name: str, arguments: dict[str, Any] | None = None):
    resolved_name = resolve_live_tool_name(name)
    async with AgentDecompileMcpClient(url=url) as client:
        return await client.call_tool(resolved_name, arguments or {})

async def record_tool(label: str, name: str, arguments: dict[str, Any] | None = None, expect_error: bool = False):
    started = time.time()
    error_text = None
    payload = None
    succeeded = False
    resolved_name = resolve_live_tool_name(name)
    try:
        payload = await call_tool_live(SERVER_INFO["url"], resolved_name, arguments or {})
        succeeded = True
    except Exception as exc:
        error_text = str(exc)
        payload = {"error": error_text}
    entry = {
        "label": label,
        "tool": resolved_name,
        "arguments": arguments or {},
        "succeeded": succeeded,
        "expect_error": expect_error,
        "matched_expectation": (not succeeded) if expect_error else succeeded,
        "duration_seconds": round(time.time() - started, 3),
        "payload": payload,
        "error": error_text,
    }
    RESULTS.append(entry)
    return entry

def extract_program_path(payload: Any) -> str | None:
    if not isinstance(payload, dict):
        return None
    for key in ["path", "programPath", "program_path", "checkedOutProgram"]:
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return value
    return None

def first_function_identifier(payload: Any) -> str | None:
    if not isinstance(payload, dict):
        return None
    functions = payload.get("functions")
    if isinstance(functions, list) and functions:
        first = functions[0]
        if isinstance(first, dict):
            for key in ["address", "name"]:
                value = first.get(key)
                if isinstance(value, str) and value.strip():
                    return value
    return None

def stop_local_server() -> None:
    global SERVER_PROCESS, SERVER_LOG_HANDLE
    if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=10)
        except subprocess.TimeoutExpired:
            SERVER_PROCESS.kill()
            SERVER_PROCESS.wait(timeout=10)
    SERVER_PROCESS = None
    if SERVER_LOG_HANDLE is not None:
        SERVER_LOG_HANDLE.close()
        SERVER_LOG_HANDLE = None

In [46]:
RESULTS.clear()
CURRENT_PROGRAM_PATH = None
SERVER_INFO = start_local_server()
show_note(
    "Starting local server",
    f"Command: <code>{html.escape(SERVER_INFO['command'])}</code><br>URL: <code>{html.escape(SERVER_INFO['url'])}</code>",
    color="#6a3fb5",
)

SERVER_READY = await wait_for_server(SERVER_INFO["url"], timeout=120)
if not SERVER_READY:
    raise RuntimeError(
        "Local server did not become ready. Check the server log tail below.\n\n" + read_server_log_tail()
    )

live_tools = await list_tools_live(SERVER_INFO["url"])
live_resources = await list_resources_live(SERVER_INFO["url"])

def tool_name(tool: Any) -> str:
    if isinstance(tool, dict):
        return str(tool.get("name", ""))
    if hasattr(tool, "name"):
        return str(tool.name)
    return str(tool)

TOOL_NAMES = sorted(name for name in (tool_name(tool) for tool in live_tools) if name)
OPEN_TOOL = resolve_live_tool_name("open-project")

display(Markdown("## Live Tool Discovery"))
as_json({
    "tool_count": len(TOOL_NAMES),
    "resource_count": len(live_resources),
    "open_tool": OPEN_TOOL,
    "tool_sample": TOOL_NAMES[:20],
})

2026-03-05 23:29:55,402 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:29:55,405 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:29:55,412 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:29:55,873 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:29:55,875 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:29:55,881 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:29:56,357 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:29:56,361 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Live Tool Discovery

<IPython.core.display.JSON object>

In [47]:
function_related_tools = [name for name in TOOL_NAMES if "function" in name]
as_json({
    "function_related_tools": function_related_tools,
    "has_list_functions": any(name in TOOL_NAMES for name in ["list-functions", "list_functions"]),
    "has_get_functions": any(name in TOOL_NAMES for name in ["get-functions", "get_functions"]),
    "has_decompile_function": any(name in TOOL_NAMES for name in ["decompile-function", "decompile_function"]),
})

<IPython.core.display.JSON object>

In [48]:
if OPEN_TOOL is None:
    raise RuntimeError("No open tool was advertised by the local server.")

open_entry = await record_tool(
    "Open local binary",
    OPEN_TOOL,
    {"path": str(PRIMARY_BINARY)},
)
project_listing_entry = await record_tool(
    "List project files",
    "list-project-files",
    {},
)
current_program_entry = await record_tool(
    "Get current program",
    "get-current-program",
    {},
)

CURRENT_PROGRAM_PATH = extract_program_path(current_program_entry["payload"])
display(Markdown("## Local Open Flow"))
as_json(
    {
        "open_succeeded": open_entry["succeeded"],
        "current_program_path": CURRENT_PROGRAM_PATH,
        "project_listing_succeeded": project_listing_entry["succeeded"],
        "open_payload": open_entry["payload"],
        "current_program_payload": current_program_entry["payload"],
    }
)

2026-03-05 23:29:57,114 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:29:57,117 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:30:02,104 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:02,607 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:02,611 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:30:02,615 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:03,123 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:03,126 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Local Open Flow

<IPython.core.display.JSON object>

In [49]:
def with_program(arguments: dict[str, Any] | None = None) -> dict[str, Any]:
    merged = dict(arguments or {})
    if CURRENT_PROGRAM_PATH and "programPath" not in merged and "program_path" not in merged:
        merged["programPath"] = CURRENT_PROGRAM_PATH
    return merged

functions_entry = await record_tool(
    "List functions sample",
    "list-functions",
    with_program({"limit": 5}),
)
search_entry = await record_tool(
    "Search symbols by name",
    "search-symbols-by-name",
    with_program({"query": "main", "max_results": 5}),
)

sample_identifier = first_function_identifier(functions_entry["payload"])
references_target = sample_identifier or "entry"
references_to_entry = await record_tool(
    "References to sample target",
    "get-references",
    with_program({"mode": "to", "target": references_target, "limit": 10}),
)

display(Markdown("## Read-Only Inspection Flows"))
as_json(
    {
        "function_identifier_used": sample_identifier,
        "functions_payload": functions_entry["payload"],
        "search_payload": search_entry["payload"],
        "references_payload": references_to_entry["payload"],
    }
)

2026-03-05 23:30:03,865 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:03,868 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:30:03,965 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:04,558 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:04,563 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:30:04,629 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:05,388 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:05,391 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Read-Only Inspection Flows

<IPython.core.display.JSON object>

In [40]:
payload_preview = str(functions_entry["payload"])
if len(payload_preview) > 500:
    payload_preview = payload_preview[:497] + "..."
as_json({
    "functions_entry_tool": functions_entry["tool"],
    "functions_entry_succeeded": functions_entry["succeeded"],
    "functions_entry_payload_preview": payload_preview,
})

<IPython.core.display.JSON object>

In [50]:
export_entry = await record_tool(
    "Export HTML",
    "export",
    with_program({
        "format": "html",
        "outputPath": str(OUTPUT_DIR / "program.html"),
    }),
)
imports_entry = await record_tool(
    "List imports",
    "list-imports",
    with_program({"limit": 5}),
)
exports_entry = await record_tool(
    "List exports",
    "list-exports",
    with_program({"limit": 5}),
)
strings_entry = await record_tool(
    "Manage strings regex",
    "manage-strings",
    with_program({
        "mode": "regex",
        "query": "Agent|Test|ELF",
        "include_referencing_functions": True,
        "limit": 20,
    }),
)
constants_entry = await record_tool(
    "Search constants",
    "search-constants",
    with_program({"mode": "specific", "value": 32, "max_results": 20}),
)

advanced_summary = {
    "export_payload": export_entry["payload"],
    "imports_payload": imports_entry["payload"],
    "exports_payload": exports_entry["payload"],
    "strings_payload": strings_entry["payload"],
    "constants_payload": constants_entry["payload"],
    "export_exists": (OUTPUT_DIR / "program.html").exists(),
}

if sample_identifier:
    info_entry = await record_tool(
        "Get function info",
        "get-functions",
        with_program({
            "identifier": sample_identifier,
            "view": "info",
            "include_callers": True,
            "include_callees": True,
        }),
    )
    decompile_entry = await record_tool(
        "Get function decompile",
        "get-functions",
        with_program({"identifier": sample_identifier, "view": "decompile"}),
    )
    disassemble_entry = await record_tool(
        "Get function disassemble",
        "get-functions",
        with_program({"identifier": sample_identifier, "view": "disassemble"}),
    )
    call_graph_entry = await record_tool(
        "Get call graph",
        "get-call-graph",
        with_program({
            "function_identifier": sample_identifier,
            "mode": "callees",
            "max_depth": 2,
        }),
    )
    references_from_entry = await record_tool(
        "References from sample function",
        "get-references",
        with_program({"mode": "from", "target": sample_identifier, "limit": 25}),
    )
    data_flow_entry = await record_tool(
        "Analyze data flow",
        "analyze-data-flow",
        with_program({
            "function_address": sample_identifier,
            "start_address": sample_identifier,
            "direction": "forward",
        }),
    )
    advanced_summary["function_detail_payloads"] = {
        "info": info_entry["payload"],
        "decompile": decompile_entry["payload"],
        "disassemble": disassemble_entry["payload"],
        "call_graph": call_graph_entry["payload"],
        "references_from": references_from_entry["payload"],
        "data_flow": data_flow_entry["payload"],
    }
else:
    advanced_summary["function_detail_payloads"] = "Skipped: no sample function identifier was available."

display(Markdown("## Extended Local Tool Flows"))
as_json(advanced_summary)

2026-03-05 23:30:06,086 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:06,093 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:30:06,111 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:06,751 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:06,753 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:30:06,757 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:07,248 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:07,251 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Extended Local Tool Flows

<IPython.core.display.JSON object>

In [51]:
version_control_probe_summary = "Skipped by widget flag."
if RUN_VERSION_CONTROL_PROBES.value:
    checkout_status_entry = await record_tool(
        "Checkout status in local mode",
        "checkout-status",
        with_program({}),
        expect_error=True,
    )
    checkout_entry = await record_tool(
        "Checkout program in local mode",
        "checkout-program",
        with_program({"exclusive": False}),
        expect_error=True,
    )
    checkin_entry = await record_tool(
        "Checkin program in local mode",
        "checkin-program",
        with_program({
            "comment": "Notebook local-mode behavior probe",
            "keep_checked_out": False,
        }),
        expect_error=True,
    )
    version_control_probe_summary = {
        "checkout_status": checkout_status_entry["payload"],
        "checkout": checkout_entry["payload"],
        "checkin": checkin_entry["payload"],
    }

display(Markdown("## Explicit Version-Control Behavior Probes"))
as_json(version_control_probe_summary)

2026-03-05 23:30:09,029 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:09,032 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:30:09,035 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:09,562 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:09,564 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-05 23:30:09,569 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:10,054 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 200 OK"
2026-03-05 23:30:10,056 [INFO] HTTP Request: POST http://127.0.0.1:4707/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Explicit Version-Control Behavior Probes

<IPython.core.display.JSON object>

## Shared-Server Workflows

 > These examples stay visible for onboarding, but they are intentionally not executed by default in this notebook.

Shared-server commands in [USAGE.md] assume external credentials and a live Ghidra shared repository. This notebook keeps them as documentation-only examples so the default experience remains local, reproducible, and read-only.

### Intended semantic rule captured by this notebook

- Local projects should remain read-only by default.
- `checkout-program` and `checkin-program` are explicit operations, not implicit side effects of opening a local binary.
- If version control is unsupported in the current context, the failure should be early and clear rather than silently promoting the project into a writable shared-server state.

In [18]:
def summarize_results() -> list[dict[str, Any]]:
    rows = []
    for entry in RESULTS:
        payload_text = json.dumps(entry["payload"], default=str)
        if len(payload_text) > 220 and not current_flags()["verbose_tool_output"]:
            payload_text = payload_text[:217] + "..."
        rows.append(
            {
                "label": entry["label"],
                "tool": entry["tool"],
                "succeeded": entry["succeeded"],
                "expect_error": entry["expect_error"],
                "matched_expectation": entry["matched_expectation"],
                "duration_seconds": entry["duration_seconds"],
                "payload_preview": payload_text,
            }
        )
    return rows

summary_rows = summarize_results()
summary_table_rows = "".join(
    "<tr>"
    f"<td style='padding:0.35rem 0.5rem'>{html.escape(str(row['label']))}</td>"
    f"<td style='padding:0.35rem 0.5rem'><code>{html.escape(str(row['tool']))}</code></td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['succeeded']}</td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['expect_error']}</td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['matched_expectation']}</td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['duration_seconds']}</td>"
    f"<td style='padding:0.35rem 0.5rem'><code>{html.escape(str(row['payload_preview']))}</code></td>"
    "</tr>"
    for row in summary_rows
 )

display(Markdown("## Execution Summary"))
display(
    HTML(
        "<table style='border-collapse:collapse; width:100%'>"
        "<thead><tr>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Label</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Tool</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Succeeded</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Expect error</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Matched expectation</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Seconds</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Payload preview</th>"
        "</tr></thead>"
        f"<tbody>{summary_table_rows}</tbody></table>"
    )
)

show_note(
    "Interpretation",
    "Rows where <code>matched_expectation</code> is false indicate either a genuine regression or a place where the desired read-only/default semantics do not match current implementation.",
    color="#8a1c1c",
)

## Execution Summary

Label,Tool,Succeeded,Expect error,Matched expectation,Seconds,Payload preview
Get functions sample,get-functions,False,False,False,4.649,"{""error"": ""Cannot connect to AgentDecompile server at http://127.0.0.1:14684/mcp/message: All connection attempts failed\n\nPlease start the AgentDecompile server first:\n\n agentdecompile-server --transport streamab..."
Search symbols by name,search-symbols-by-name,False,False,False,4.514,"{""error"": ""Cannot connect to AgentDecompile server at http://127.0.0.1:14684/mcp/message: All connection attempts failed\n\nPlease start the AgentDecompile server first:\n\n agentdecompile-server --transport streamab..."
References to sample target,get-references,False,False,False,4.528,"{""error"": ""Cannot connect to AgentDecompile server at http://127.0.0.1:14684/mcp/message: All connection attempts failed\n\nPlease start the AgentDecompile server first:\n\n agentdecompile-server --transport streamab..."
Open local binary,open_project,True,False,True,7.457,"{""content"": [{""type"": ""text"", ""text"": ""## Project (import)\n\n**operation:** import\n**importedFrom:** C:\\GitHub\\agentdecompile\\tests\\fixtures\\test_x86_64\n**filesDiscovered:** 1\n**filesImported:** 1\n\n### Impo..."
List project files,list_project_files,True,False,True,0.484,"{""content"": [{""type"": ""text"", ""text"": ""## Project Files\n\n**Folder:** `/`\n**Count:** 0\n\n\n### About This Tool\n\nLists files in the current Ghidra project directory.""}], ""isError"": false}"
Get current program,get_current_program,True,False,True,0.615,"{""content"": [{""type"": ""text"", ""text"": ""## Current Program\n\n**Name:** `test_x86_64`\n**Path:** ``\n**Language:** x86:LE:64:default\n**Compiler:** gcc\n**Image Base:** ``\n**Functions:** 3\n**Symbols:** 0\n\n### About..."
Get functions sample,decompile_function,True,False,True,0.529,"{""content"": [{""type"": ""text"", ""text"": ""## Error\n\n> **function or addressOrSymbol required**\n\n**Tool:** `decompile_function`\n\n### About This Tool\n\nConverts machine code into C-like pseudocode using Ghidra's dec..."
Search symbols by name,search-symbols-by-name,True,False,True,0.597,"{""content"": [{""type"": ""text"", ""text"": ""## Symbol Listing\n\nShowing **2** of **2** results (offset 0).\n\n| Name | Address | Type | Namespace |\n| --- | --- | --- | --- |\n| _main | 1000004b0 | Label | Global |\n| s__..."
References to sample target,get_references,True,False,True,0.53,"{""content"": [{""type"": ""text"", ""text"": ""## Getreferences (to)\n\n**mode:** to\n**target:** 1000004b0\n\n### References\n\n| fromAddress | toAddress | type | function |\n| --- | --- | --- | --- |\n| Entry Point | 100000..."
Export HTML,export,True,False,True,0.504,"{""content"": [{""type"": ""text"", ""text"": ""## Export Results\n\n**Format:** html\n**Output Path:** `C:\\GitHub\\agentdecompile\\tmp\\usage_validation\\usage_validation_zilg1cud\\exports\\program.html`\n**Status:** Success..."


In [52]:
def compact_payload(payload: Any) -> Any:
    if isinstance(payload, dict):
        return {key: compact_payload(value) for key, value in list(payload.items())[:10]}
    if isinstance(payload, list):
        return [compact_payload(item) for item in payload[:5]]
    return payload

compact_summary = {
    "open_entry": {
        "tool": open_entry["tool"],
        "succeeded": open_entry["succeeded"],
        "payload": compact_payload(open_entry["payload"]),
    },
    "current_program_entry": {
        "tool": current_program_entry["tool"],
        "succeeded": current_program_entry["succeeded"],
        "payload": compact_payload(current_program_entry["payload"]),
    },
    "functions_entry": {
        "tool": functions_entry["tool"],
        "succeeded": functions_entry["succeeded"],
        "payload": compact_payload(functions_entry["payload"]),
    },
    "export_entry": {
        "tool": export_entry["tool"],
        "succeeded": export_entry["succeeded"],
        "payload": compact_payload(export_entry["payload"]),
    },
    "version_control_rows": [
        {
            "label": entry["label"],
            "tool": entry["tool"],
            "succeeded": entry["succeeded"],
            "expect_error": entry["expect_error"],
            "matched_expectation": entry["matched_expectation"],
            "payload": compact_payload(entry["payload"]),
        }
        for entry in RESULTS
        if "checkout" in entry["tool"] or "checkin" in entry["tool"]
    ],
}
as_json(compact_summary)

<IPython.core.display.JSON object>

In [53]:
summary_artifact = SESSION_DIR / "compact_summary.json"
summary_artifact.write_text(json.dumps(compact_summary, indent=2, default=str), encoding="utf-8")
print(summary_artifact)

C:\GitHub\agentdecompile\tmp\usage_validation\usage_validation_zilg1cud\compact_summary.json


In [54]:
stop_local_server()
show_note(
    "Cleanup complete",
    f"Temporary workspace retained for inspection at <code>{html.escape(str(SESSION_DIR))}</code>.",
    color="#1f6f43",
)
print(read_server_log_tail(40))

{"jsonrpc":"2.0","method":"_log","params":{"message":"INFO:     127.0.0.1:4759 - \"POST /mcp/message HTTP/1.1\" 202 Accepted"}}
{"jsonrpc":"2.0","method":"_log","params":{"message":"INFO:     127.0.0.1:4759 - \"POST /mcp/message HTTP/1.1\" 200 OK"}}
{"jsonrpc":"2.0","method":"_log","params":{"message":"INFO:     127.0.0.1:4760 - \"POST /mcp/message HTTP/1.1\" 200 OK"}}
{"jsonrpc":"2.0","method":"_log","params":{"message":"INFO:     127.0.0.1:4760 - \"POST /mcp/message HTTP/1.1\" 202 Accepted"}}
{"jsonrpc":"2.0","method":"_log","params":{"message":"INFO:     127.0.0.1:4760 - \"POST /mcp/message HTTP/1.1\" 200 OK"}}
{"jsonrpc":"2.0","method":"_log","params":{"message":"INFO:     127.0.0.1:4763 - \"POST /mcp/message HTTP/1.1\" 200 OK"}}
{"jsonrpc":"2.0","method":"_log","params":{"message":"INFO:     127.0.0.1:4763 - \"POST /mcp/message HTTP/1.1\" 202 Accepted"}}
{"jsonrpc":"2.0","method":"_log","params":{"message":"INFO:     127.0.0.1:4763 - \"POST /mcp/message HTTP/1.1\" 200 OK"}}
{"json